# test_api.ipynb

Test notebook for the `api_server.py` FastAPI server running at `http://127.0.0.1:3000`.

**Before running this notebook**, start the server in a terminal:
```bash
python api_server.py
```

## Endpoint
| Method | URL | Description |
|---|---|---|
| GET | `/` | Health check |
| POST | `/query` | Execute a SELECT SQL query |

## Request format
```json
{ "sql": "SELECT * FROM suppliers LIMIT 5" }
```

## Response format (success)
```json
{
  "status": "ok",
  "rows_returned": 5,
  "columns": ["supplier_id", "supplier_name", ...],
  "data": [ { "supplier_id": 1, "supplier_name": "...", ... }, ... ]
}
```

## Response format (rejected / error)
```json
{ "status": "rejected" | "error", "message": "<reason>" }
```

In [ ]:
import requests
import pandas as pd
import json

BASE_URL = "http://127.0.0.1:3000"

def query(sql: str, label: str = "") -> pd.DataFrame | dict:
    """
    Helper: POST a SQL string to /query and pretty-print the result.
    - On success (status=ok): returns a pandas DataFrame.
    - On rejection/error: prints the message and returns the raw dict.
    """
    if label:
        print(f"\n{'='*60}")
        print(f"  {label}")
        print(f"{'='*60}")
    
    resp = requests.post(f"{BASE_URL}/query", json={"sql": sql})
    body = resp.json()
    
    print(f"HTTP {resp.status_code} | status: {body.get('status')}")
    
    if body.get("status") == "ok":
        df = pd.DataFrame(body["data"], columns=body["columns"])
        print(f"Rows returned: {body['rows_returned']}")
        display(df)
        return df
    else:
        print(f"Message: {body.get('message')}")
        return body

## 1. Health Check

In [ ]:
# GET / — confirm server is up
resp = requests.get(f"{BASE_URL}/")
print(f"HTTP {resp.status_code}")
print(resp.json())

## 2. Valid SELECT Queries

In [ ]:
# Row counts across all tables
query("""
    SELECT 'suppliers'  AS tbl, COUNT(*) AS rows FROM suppliers  UNION ALL
    SELECT 'products',          COUNT(*)          FROM products   UNION ALL
    SELECT 'inventory',         COUNT(*)          FROM inventory  UNION ALL
    SELECT 'orders',            COUNT(*)          FROM orders     UNION ALL
    SELECT 'sales_items',       COUNT(*)          FROM sales_items
""", label="Row counts per table")

In [ ]:
# Fetch all suppliers
query("SELECT * FROM suppliers", label="All suppliers")

In [ ]:
# Products grouped by category
query("""
    SELECT category, COUNT(*) AS product_count
    FROM products
    GROUP BY category
    ORDER BY product_count DESC
""", label="Products by category")

In [ ]:
# Top 5 products by revenue
query("""
    SELECT p.product_name,
           ROUND(SUM(si.line_sale_amount), 2) AS total_revenue,
           SUM(si.quantity) AS total_units_sold
    FROM sales_items si
    JOIN products p ON si.product_id = p.product_id
    GROUP BY si.product_id
    ORDER BY total_revenue DESC
    LIMIT 5
""", label="Top 5 products by revenue")

In [ ]:
# Items below reorder point
query("""
    SELECT i.product_id, p.product_name, i.quantity_on_hand, i.reorder_point
    FROM inventory i
    JOIN products p ON i.product_id = p.product_id
    WHERE i.quantity_on_hand < i.reorder_point
    ORDER BY i.quantity_on_hand ASC
    LIMIT 10
""", label="Stock below reorder point (top 10)")

In [ ]:
# Revenue per supplier (cross-table join)
query("""
    SELECT s.supplier_name,
           COUNT(DISTINCT p.product_id) AS products,
           ROUND(SUM(si.line_sale_amount), 2) AS total_revenue
    FROM suppliers s
    JOIN products p     ON s.supplier_id = p.supplier_id
    JOIN sales_items si ON p.product_id  = si.product_id
    GROUP BY s.supplier_id
    ORDER BY total_revenue DESC
""", label="Revenue per supplier")

## 3. Blocked Queries (expect HTTP 403 rejected)

In [ ]:
# Attempt DROP TABLE — must be rejected
query("DROP TABLE suppliers", label="BLOCKED: DROP TABLE")

In [ ]:
# Attempt DELETE — must be rejected
query("DELETE FROM products WHERE product_id = 1", label="BLOCKED: DELETE")

In [ ]:
# Attempt UPDATE — must be rejected
query("UPDATE suppliers SET is_active = 0 WHERE supplier_id = 1", label="BLOCKED: UPDATE")

In [ ]:
# Attempt INSERT — must be rejected
query("INSERT INTO suppliers (supplier_code, supplier_name) VALUES ('X', 'Hacker')", label="BLOCKED: INSERT")

In [ ]:
# Attempt statement stacking via semicolon — DROP is still caught by keyword check
query("SELECT * FROM suppliers; DROP TABLE suppliers", label="BLOCKED: Statement stacking")

In [ ]:
# Attempt PRAGMA write — must be rejected
query("PRAGMA foreign_keys = OFF", label="BLOCKED: PRAGMA write")

## 4. Malformed / Invalid Inputs (expect HTTP 400 or 422 error)

In [ ]:
# SELECT from a non-existent table — SQL error
query("SELECT * FROM nonexistent_table", label="ERROR: Non-existent table")

In [ ]:
# Malformed SQL syntax
query("SELECT FROM WHERE", label="ERROR: Malformed SQL syntax")

In [ ]:
# Empty string input — validation error
query("", label="ERROR: Empty query string")

In [ ]:
# Missing 'sql' field entirely — Pydantic 422 validation error
resp = requests.post(f"{BASE_URL}/query", json={"query": "SELECT 1"})
print(f"HTTP {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

In [ ]:
# SELECT a column that doesn't exist
query("SELECT nonexistent_column FROM suppliers", label="ERROR: Non-existent column")